# 🔄 The Pivot — GRPO Training on Google Colab
**Meta PyTorch OpenEnv Hackathon 2026**

Trains **Qwen2.5-3B-Instruct** with GRPO + Adaptive Curriculum on the Pivot environment.

**Before you start:** Runtime → Change runtime type → **T4 GPU**

## Cell 1 — Check GPU

In [ ]:
import subprocess, torch
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', r.stdout.strip() or '❌ No GPU — go to Runtime → Change runtime type → T4')
print(f'PyTorch CUDA: {torch.cuda.is_available()}')

## Cell 2 — Install packages (~3 min)

In [ ]:
%%capture
!pip install -q openenv-core
!pip install -q "transformers>=4.45.0"
!pip install -q "peft>=0.13.0"
!pip install -q "bitsandbytes>=0.43.0"
!pip install -q "accelerate>=0.34.0"
!pip install -q wandb pydantic numpy python-dotenv
print('done')

## Cell 3 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE_DIR = '/content/drive/MyDrive/the_pivot_model'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'✅ Model will save to: {SAVE_DIR}')

## Cell 4 — Clone project from GitHub

In [ ]:
import os, sys

# Remove old copy if it exists, then clone fresh
!rm -rf /content/meta_scaler
!git clone https://github.com/Harshit-Makraria/meta_scaler /content/meta_scaler

PROJECT_ROOT = '/content/meta_scaler'
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
os.environ['WANDB_MODE'] = 'disabled'  # disable W&B inside the env itself

# Verify
try:
    from models import PivotAction, ActionType, PivotObservation
    from server.pivot_environment import ThePivotEnvironment
    from server.prompt_encoder import encode_to_messages
    from training.curriculum import AdaptiveCurriculum, DIFFICULTY_LADDER
    print('✅ All imports working!')
except ImportError as e:
    print(f'❌ Import error: {e}')

## Cell 5 — W&B Login

In [ ]:
import wandb
wandb.login()   # paste your key from wandb.ai/settings when prompted
WANDB_PROJECT = 'models-nexica-ai'
print('✅ W&B ready')

## Cell 6 — Load model (memory-optimised)
Qwen2.5-3B with 4-bit quantization + LoRA rank 8 to fit comfortably on T4.

In [ ]:
import os, gc, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

# ── Critical: tell PyTorch not to fragment GPU memory ──────────────────────
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
DEVICE = 'cuda'
MAX_SEQ_LEN = 512   # keep prompts short to save VRAM
MAX_NEW_TOKENS = 8  # action is one word, 8 tokens is enough

print(f'Loading {MODEL_NAME} with 4-bit quantization...')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)

# LoRA rank 8 (was 16) — halves the trainable parameter memory
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj'],  # only q+v (not k+o) — less memory
    bias='none',
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

gc.collect()
torch.cuda.empty_cache()
print(f'\n✅ Model ready!')
print(f'GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB used / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB total')

## Cell 7 — Training helpers

In [ ]:
import re, random, numpy as np
from models import PivotAction, ActionType, PivotObservation
from server.pivot_environment import ThePivotEnvironment
from server.prompt_encoder import encode_to_messages

ACTION_MAP = {
    'execute':   ActionType.EXECUTE,
    'pivot':     ActionType.PIVOT,
    'research':  ActionType.RESEARCH,
    'fundraise': ActionType.FUNDRAISE,
    'hire':      ActionType.HIRE,
    'cut_costs': ActionType.CUT_COSTS,
    'cut':       ActionType.CUT_COSTS,
}


def _parse_action(text: str) -> ActionType:
    word = re.sub(r'[^a-z_]', '', text.lower().split()[0]) if text.strip() else 'execute'
    return ACTION_MAP.get(word, ActionType.EXECUTE)


@torch.no_grad()
def generate_action(obs: PivotObservation) -> tuple[str, ActionType]:
    """Run the LLM → return (raw_text, ActionType). No gradients."""
    messages = encode_to_messages(obs)
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=MAX_SEQ_LEN).to(DEVICE)
    out = model.generate(
        **inputs, max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True, temperature=0.7, top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
    decoded = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    # Free immediately
    del inputs, out
    return decoded, _parse_action(decoded)


def get_log_prob(prompt_text: str, completion_text: str) -> torch.Tensor:
    """Log prob of completion given prompt. Called with gradients enabled."""
    full = prompt_text + completion_text
    inp = tokenizer(full, return_tensors='pt', truncation=True, max_length=MAX_SEQ_LEN).to(DEVICE)
    p_len = tokenizer(prompt_text, return_tensors='pt', truncation=True,
                      max_length=MAX_SEQ_LEN)['input_ids'].shape[1]

    logits = model(**inp).logits  # (1, seq, vocab)
    comp_logits = logits[0, p_len - 1:-1, :]  # completion slice
    comp_ids    = inp['input_ids'][0, p_len:]  # target ids

    del inp, logits

    if comp_ids.shape[0] == 0:
        return torch.tensor(0.0, requires_grad=True, device=DEVICE)

    lp = torch.nn.functional.log_softmax(comp_logits, dim=-1)
    return lp[range(len(comp_ids)), comp_ids].mean()


def run_episode(scenario: dict) -> list[dict]:
    """Full 60-step rollout → list of step dicts."""
    env = ThePivotEnvironment(scenario=scenario)
    obs = env.reset()
    steps = []
    for _ in range(60):
        prompt = tokenizer.apply_chat_template(
            encode_to_messages(obs), tokenize=False, add_generation_prompt=True
        )
        # Truncate prompt to MAX_SEQ_LEN chars (rough but fast)
        if len(prompt) > MAX_SEQ_LEN * 4:
            prompt = prompt[-(MAX_SEQ_LEN * 4):]

        completion, action_type = generate_action(obs)
        next_obs = env.step(PivotAction(action_type=action_type))
        steps.append({
            'prompt':     prompt,
            'completion': completion,
            'action':     action_type.value,
            'reward':     float(next_obs.reward or 0),
            'step':       next_obs.step,
            'runway':     next_obs.runway_remaining,
        })
        obs = next_obs
        if obs.done:
            break
    return steps


print('✅ Helpers ready!')

## Cell 8 — Training config
Memory-safe settings for T4 (14.5 GB VRAM).

In [ ]:
CONFIG = {
    'n_episodes':        150,   # training episodes
    'G':                 2,     # GRPO group size — 2 completions per step (was 4, OOM)
    'lr':                5e-5,
    'grad_clip':         1.0,
    'grpo_steps_sample': 4,     # steps sampled per episode for update (was 20, OOM)
    'save_every':        25,
    'log_every':         5,
}
print('Config:')
for k, v in CONFIG.items(): print(f'  {k:25s} {v}')

## Cell 9 — GRPO update function + optimizer

In [ ]:
from torch.optim import AdamW
from training.curriculum import AdaptiveCurriculum, DIFFICULTY_LADDER

optimizer  = AdamW(model.parameters(), lr=CONFIG['lr'])
curriculum = AdaptiveCurriculum(seed=42)


def grpo_update(episode_steps: list[dict]) -> dict:
    """
    GRPO policy-gradient update.
    Samples a few steps from the episode, generates G completions each,
    normalises rewards within the group, backprops.
    """
    model.train()
    optimizer.zero_grad()
    total_loss = 0.0
    n_updates  = 0

    sampled = random.sample(episode_steps,
                            min(CONFIG['grpo_steps_sample'], len(episode_steps)))

    for sd in sampled:
        prompt = sd['prompt']
        completions = [sd['completion']]        # 1st: the actual rollout completion
        rewards     = [sd['reward']]

        # Sample G-1 alternative completions (no grad, just for reward diversity)
        for _ in range(CONFIG['G'] - 1):
            with torch.no_grad():
                inp = tokenizer(prompt, return_tensors='pt',
                                truncation=True, max_length=MAX_SEQ_LEN).to(DEVICE)
                out = model.generate(**inp, max_new_tokens=MAX_NEW_TOKENS,
                                     do_sample=True, temperature=1.1, top_p=0.95,
                                     pad_token_id=tokenizer.eos_token_id)
                alt = tokenizer.decode(out[0][inp['input_ids'].shape[1]:],
                                       skip_special_tokens=True).strip()
                del inp, out
            # Proxy reward: same as rollout + tiny noise (env can't be rewound)
            completions.append(alt)
            rewards.append(sd['reward'] + random.gauss(0, 1.5))

        # Normalise within group
        r_mean = np.mean(rewards)
        r_std  = np.std(rewards) + 1e-8
        normed = [(r - r_mean) / r_std for r in rewards]

        # Policy gradient loss
        step_loss = torch.tensor(0.0, device=DEVICE)
        for comp, nr in zip(completions, normed):
            lp = get_log_prob(prompt, comp)
            step_loss = step_loss - lp * nr
            # Free cache after every log-prob to prevent fragmentation
            torch.cuda.empty_cache()

        step_loss = step_loss / CONFIG['G']
        step_loss.backward()
        total_loss += step_loss.item()
        n_updates  += 1

        # Free after each step
        del step_loss
        gc.collect()
        torch.cuda.empty_cache()

    torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
    optimizer.step()
    optimizer.zero_grad()
    model.eval()

    gc.collect()
    torch.cuda.empty_cache()

    return {'loss': total_loss / max(n_updates, 1)}


print('✅ Optimizer + GRPO update ready!')

## Cell 10 — Start W&B run + train!

In [ ]:
import gc

# Init W&B
run = wandb.init(project=WANDB_PROJECT, name='grpo-qwen2.5-3b-pivot',
                 config=CONFIG, tags=['grpo','qwen2.5-3b','pivot','hackathon'])
print(f'W&B run: {run.get_url()}')
print(f'Starting tier {curriculum.current_tier+1}/5: {DIFFICULTY_LADDER[curriculum.current_tier]}')
print('─' * 70)

episode_rewards, episode_lengths, all_losses = [], [], []
model.eval()

for ep in range(1, CONFIG['n_episodes'] + 1):

    # ── Rollout ──────────────────────────────────────────────────────────────
    scenario      = curriculum.sample_scenario()
    scenario_name = scenario.get('name', 'default')
    steps         = run_episode(scenario)

    ep_reward   = sum(s['reward'] for s in steps)
    ep_length   = len(steps)
    survived    = ep_length >= 60
    pivot_count = sum(1 for s in steps if s['action'] == 'PIVOT')

    episode_rewards.append(ep_reward)
    episode_lengths.append(ep_length)

    # ── GRPO update ──────────────────────────────────────────────────────────
    loss_stats = grpo_update(steps)
    all_losses.append(loss_stats['loss'])

    # ── Curriculum ───────────────────────────────────────────────────────────
    curriculum.record_result(ep_reward, survived)

    # ── Log ──────────────────────────────────────────────────────────────────
    gpu_used = torch.cuda.memory_allocated() / 1e9
    print(
        f'Ep {ep:3d}/{CONFIG["n_episodes"]} | '
        f'{scenario_name:18s} | T{curriculum.current_tier+1} | '
        f'steps {ep_length:3d} | reward {ep_reward:+7.1f} | '
        f'loss {loss_stats["loss"]:6.4f} | pivots {pivot_count} | '
        f'{"SURVIVED" if survived else "died"} | GPU {gpu_used:.1f}GB'
    )

    if ep % CONFIG['log_every'] == 0:
        wandb.log({
            'episode':            ep,
            'ep_reward':          ep_reward,
            'mean_reward_20ep':   np.mean(episode_rewards[-20:]),
            'ep_length':          ep_length,
            'survived':           int(survived),
            'pivot_count':        pivot_count,
            'loss':               loss_stats['loss'],
            'curriculum_tier':    curriculum.current_tier + 1,
            'gpu_gb':             gpu_used,
            'scenario':           scenario_name,
        })

    # ── Curriculum advance ───────────────────────────────────────────────────
    if curriculum.should_advance() and curriculum.advance_tier():
        new = DIFFICULTY_LADDER[curriculum.current_tier]
        print(f'\n🎓 CURRICULUM ADVANCE → Tier {curriculum.current_tier+1}/5: {new}\n')
        wandb.log({'curriculum_advance': curriculum.current_tier, 'episode': ep})

    # ── Checkpoint ───────────────────────────────────────────────────────────
    if ep % CONFIG['save_every'] == 0:
        ckpt = f'{SAVE_DIR}/checkpoint_ep{ep}'
        model.save_pretrained(ckpt)
        tokenizer.save_pretrained(ckpt)
        print(f'💾 Saved checkpoint → {ckpt}')


print('\n' + '='*70)
print('✅ TRAINING COMPLETE!')
print(f'Mean reward (last 20): {np.mean(episode_rewards[-20:]):.1f}')
print(f'Best reward:           {max(episode_rewards):.1f}')
print(f'Survival rate:         {sum(l>=60 for l in episode_lengths)/len(episode_lengths):.0%}')
print('='*70)

## Cell 11 — Save final model

In [ ]:
FINAL = f'{SAVE_DIR}/final_model'
model.save_pretrained(FINAL)
tokenizer.save_pretrained(FINAL)
print(f'✅ Saved to {FINAL}')
wandb.finish()
print('✅ W&B run closed')

## Cell 12 — Evaluate: trained vs baselines

In [ ]:
from training.baseline_agent import RandomAgent, StubbornAgent, PanicAgent, run_episodes
from training.curriculum import AdaptiveCurriculum, DIFFICULTY_LADDER
import json

N_EVAL = 20
c = AdaptiveCurriculum()

class TrainedAgent:
    name = 'trained_llm'
    def act(self, obs):
        _, at = generate_action(obs)
        return PivotAction(action_type=at)

agents = [RandomAgent(), StubbornAgent(), PanicAgent(), TrainedAgent()]
print(f'{"Scenario":22s} | {"Agent":12s} | Reward   | Survival | PivotRate')
print('-' * 68)
all_results = []
for name in DIFFICULTY_LADDER:
    sc = c._all_scenarios.get(name)
    if not sc: continue
    for agent in agents:
        r = run_episodes(agent, sc, N_EVAL)
        all_results.append(r)
        print(f'{name:22s} | {agent.name:12s} | {r["mean_reward"]:+7.1f} | {r["survival_rate"]:6.0%}   | {r["pivot_rate"]:6.0%}')

with open(f'{SAVE_DIR}/eval_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print('\n✅ Eval done — results saved to Drive')